# Chapter 11: the new SKU business case and its tornado

Kestrel Lager, 12 x 650 ml, new to synthetic State S. Monthly cash over 36 months, discounted at 12 percent.
Change one input at a time. Every assumption is the chapter's and replaceable.

In [ ]:
import sys
sys.path.append('..')
import pandas as pd
import matplotlib.pyplot as plt
from calculator import inr, r2
import math
from scipy.optimize import brentq

In [ ]:
def mrp_bottle(share, dmc=982.80, fee=15.60):
    duty = r2(share * dmc); landed = r2(dmc + duty + fee)
    shelf = r2(r2(landed * 1.08) * 1.15)
    return 5 * math.ceil(round(shelf / 12, 6) / 5)

def sku_case(hl=2000, glass=18.0, start=4, lag=1, share=1.10, elasticity=None, loaded=False, months=36):
    """Monthly cash: duty, fee, beer, packaging and freight go out at dispatch; the buyer repays `lag` months later."""
    dmc, fee, beer, freight, overhead = 982.80, 15.60, 224.64, 49.14, 147.42
    duty = r2(share * dmc)
    cash_out = duty + fee + beer + 12 * glass + 30 + freight + (overhead if loaded else 0)
    cash_in = dmc + duty + fee
    if elasticity is not None:
        hl *= 1 + elasticity * (mrp_bottle(share) - mrp_bottle(1.10)) / mrp_bottle(1.10)
    per_month = hl / 0.078 / 12
    cash = [0.0] * (months + lag + 2)
    for m in range(start, months + 1):
        cases = per_month * (0.5 if m < start + 6 else 1)
        cash[m] -= cases * cash_out
        cash[m + lag] += cases * cash_in
    cash[start] -= 20_00_000                                  # launch spend at first dispatch
    for m in (start, start + 12, start + 24):
        cash[m] -= 25_000 if m <= months else 0              # registration each year
    for m in range(start + 12, months + 1):
        cash[m] -= 10_00_000 / 12                            # support from a year after first dispatch
    cum = pd.Series(cash).cumsum()
    return dict(total=sum(cash), npv=sum(c / 1.12 ** (t / 12) for t, c in enumerate(cash)),
                worst=cum.min(), positive_month=int(cum[cum.index > start].gt(0).idxmax()), cash=cash)

base = sku_case()
print('undiscounted INR', inr(base['total'], 0), '; NPV INR', inr(base['npv'], 0), f"({base['npv'] / 1e5:.2f} lakh)")
print(f"worst cash {base['worst'] / 1e5:.2f} lakh; cumulative cash positive in month {base['positive_month']}")

In [ ]:
drivers = {
    'Volume, steady state': (dict(hl=1500), dict(hl=2500)),
    'Glass price': (dict(glass=22.50), dict(glass=13.50)),
    'Listing delay': (dict(start=7), dict(start=2)),
    'Duty, with elasticity of minus 1': (dict(share=1.20, elasticity=-1), dict(share=1.00, elasticity=-1)),
    'Route': (dict(lag=2), None)}
tornado = pd.DataFrame({k: [sku_case(**lo)['npv'] / 1e5, sku_case(**hi)['npv'] / 1e5 if hi else None]
                        for k, (lo, hi) in drivers.items()}, index=['low', 'high']).T.round(2)
stacked = sku_case(hl=1500, glass=22.50, start=7, lag=2)
route_c = sku_case(lag=2)
loaded = sku_case(loaded=True)
duty_no_elasticity = sku_case(share=1.20)
print(f"stacked bad case {stacked['npv'] / 1e5:.2f} lakh, worst cash {stacked['worst'] / 1e5:.2f}; Route C worst cash {route_c['worst'] / 1e5:.2f}")
print(f"fully loaded NPV {loaded['npv'] / 1e5:.2f} lakh; duty at 120 percent with no volume change costs {(base['npv'] - duty_no_elasticity['npv']) / 1e5:.2f} lakh")
tornado

In [ ]:
swing = (tornado.high.fillna(base['npv'] / 1e5) - tornado.low).sort_values()
fig, ax = plt.subplots(figsize=(8, 3))
for i, k in enumerate(swing.index):
    lo, hi = tornado.loc[k, 'low'], tornado.loc[k, 'high']
    ax.barh(i, base['npv'] / 1e5 - lo, left=lo, color='tab:green')
    if pd.notna(hi):
        ax.barh(i, hi - base['npv'] / 1e5, left=base['npv'] / 1e5, color='tab:olive')
ax.set_yticks(range(len(swing))); ax.set_yticklabels(swing.index); ax.axvline(base['npv'] / 1e5, color='k', ls='--')
ax.set_xlabel('NPV at 12 percent over 36 months, INR lakh (synthetic)'); plt.show()

## The one pager's other lines

In [ ]:
zero_npv_hl = brentq(lambda h: sku_case(hl=h)['npv'], 1, 2000)
zero_npv_loaded_hl = brentq(lambda h: sku_case(hl=h, loaded=True)['npv'], 1, 2000)
per_month = 2000 / 0.078 / 12
funding_a, funding_c = r2(1616.46 * 0.12 * 30 / 365), r2(1616.46 * 0.12 * 60 / 365)
contrib_hl = {s: r2(c / h) for s, c, h in [('650 ml bottle', 463.02, 0.078), ('330 ml bottle', 425.72, 0.0792), ('330 ml can', 397.52, 0.0792)]}
print(f'NPV is zero at {zero_npv_hl:.0f} hL incremental ({zero_npv_hl / 2000:.0%} of plan) and {zero_npv_loaded_hl:.1f} hL fully loaded')
print('cases a month at full rate', round(per_month, 2), '; cash tied up', round(per_month * 1616.46 / 1e5, 1), 'lakh on Route A,', round(2 * per_month * 1616.46 / 1e5, 1), 'on Route C')
print('funding a case', funding_a, 'Route A,', funding_c, 'Route C; a month of contribution', round(per_month * 463.02 / 1e5, 2), 'lakh')
print('contribution a hL', contrib_hl, '; a moved hL gains', r2(contrib_hl['650 ml bottle'] - contrib_hl['330 ml bottle']))